In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import MultipleLocator, FixedLocator
import seaborn as sns
import plotly.express as px
from scipy.stats import norm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import itertools


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
from sklearn.impute import SimpleImputer

import sys
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk

import json

import time

from flax import nnx
import jax.numpy as jnp
import jax
import math 
import sys
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FixedLocator
import numpy as np
import pandas as pd

In [ ]:
def plot(L,IT,str_nets,wT,l_info,ex_n,
         exact_gs_energy,
         iters_Jastrow_ds,energy_Jastrow_ds,iters_Net_ds,energy_Net_ds, 
         show,path_img):
    wfig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True); x_pos = 0.4; y_pos = 0.94
    w = int(wT);  
    f_wt = ftheta(w)
    l_tp = []; l_tp.append(f_wt)
    # ========== Plotting ==========
    # Plot RBM
    ax.plot(iters_Net_ds, energy_Net_ds, label=f'{l_info[0]}({energy_Net_ds[-1]:.3f})', 
            linestyle=linestyles[0], marker=markers[0], 
            color=colors[0], markersize=5)

    # Plot Jastrow
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=f'JASTROW ({energy_Jastrow_ds[-1]:.3f})', 
            linestyle=linestyles[1], marker=markers[1], 
            color=colors[1], markersize=5)
    if exact_gs_energy != 0:
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Axis configurations
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    
    ax.set_ylabel("Energia")
    ax.set_xlabel("Interações")
    
    ax.text(x_pos, y_pos, f'(a) $L={L}$ | {l_tp[0]}',transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)
    # Save and show
    create(path_img)
    path_img = path_img +  f"EGS_L_{L}_IT_{IT}_W_{wT}_J_{ex_n}_{str_nets}.png"
    print(path_img)
    plt.savefig(path_img, dpi=300, bbox_inches='tight')
    if show == 1:
        plt.show()
    plt.close()

In [ ]:
def plot_jobs(L,IT,str_nets,wT,exact_gs_energy, 
              df,T,show,path_img):
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True)
    x_pos = 0.4
    y_pos = 0.94
    i = 0

    x1 = df["id"]
    egs = df["F_Egs"]

    media_val  = egs.mean()
    
    median_val = egs.median()

    moda_val   = egs.mode()

    # Calcular a diferença absoluta entre cada moda e a mediana
    dif = abs(moda_val - median_val)

    # Selecionar a moda cuja diferença é mínima
    moda_f_val = moda_val[dif.idxmin()]

    line_color = "#1f77b4"   # azul padrão para curva principal
    exact_color = "#d62728"  # vermelho escuro (exato)
    median_color = "#2ca02c" # verde
    mean_color = "#9467bd"   # roxo
    mode_color = "#ff7f0e"   # laranja

    ax.plot(
        x1, egs,
        label=r"$E_{gs}$",
        linestyle=linestyles[i],
        marker=markers[i],
        color=line_color,
        markersize=5,
        alpha=0.7
    )
    i = i + 1
    ax.axhline(
        y=exact_gs_energy,
        color=exact_color,
        linestyle='--',
        marker=markers[i],
        linewidth=1.5,
        label=fr"$E_{{exact}}$      = {exact_gs_energy:.4f}"

    )
    i = i + 2
    ax.axhline(
        y=median_val,
        color=median_color,
        linestyle='--',
        marker=markers[i],
        linewidth=1.5,
        label=fr"Mediana = {median_val:.4f}"
    )
    i = i + 1
    ax.axhline(
        y=media_val,
        color=mean_color,
        linestyle='--',
        marker=markers[i],
        linewidth=1.5,
        label=fr"Média     = {media_val:.4f}"
    )
    i = i + 1
    ax.axhline(
        y=moda_f_val,
        color=mode_color,
        linestyle='--',
        marker=markers[i],
        linewidth=1.5,
        label=fr"Moda      = {median_val:.4f}"
    )

    
    ax.set_ylabel(r"Energy", fontsize=12)
    ax.set_xlabel(r"Jobs", fontsize=12)

    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_minor_locator(FixedLocator([]))


    w    = int(wT);  
    f_wt = ftheta(w)
    ax.text(x_pos, y_pos, f'(a) $L={ll}$ | {f_wt}', transform=ax.transAxes, 
            fontsize=12, verticalalignment='top')

    ax.legend(
        fontsize=9, loc='upper left',
        bbox_to_anchor=(1.02, 1),
        frameon=True, framealpha=1,
        edgecolor='black'
    )

    plt.tight_layout(rect=[0, 0, 0.85, 1])  # espaço para legenda

    create(path_img)
    path_img = path_img +  f"EGS_JOBS_TP_{T}_L_{L}_IT_{IT}_W_{wT}_{str_nets}.png"
    plt.savefig(path_img, dpi=300, bbox_inches='tight')
    if show == 1:
        plt.show()
    plt.close()    